# 计算个股日收益率的月方差、偏度和峰度

In [1]:
## 导入NumPy、Pandas、Wind
import numpy as np
import pandas as pd
from scipy import stats
from WindPy import w
import datetime

## 筛选非ST股票代码

In [2]:
## 使用Wind下载全部A股公司的上市板与上市日期数据
## 使用excel left函数筛选中小板股票更改boardtype，以UTF-8 保存csv文件
## 主板、中小板、创业板、科创板共有4221家公司，其中主板公司2092家，中小板965家，创业板920家，科创板244家
corp = pd.read_csv('Raw Transaction Data/corp_info.csv', encoding = "utf-8")
corp.columns = ['codes', 'ipo_date', 'boardtype']
corp['ipo_date'] = pd.to_datetime(corp['ipo_date'])

In [8]:
## 保留18年以前上市的公司
## 共有公司3535家，主板公司1888家，中小板914家，创业板933家
corp_bf18 = corp[corp['ipo_date'] < '2019-01-01']
corp_bf18 = corp_bf18.reset_index(drop = True)
corp_bf18 = corp_bf18[['codes', 'boardtype']]

## 退市与ST公司

In [9]:
## 退市公司信息通过wind沪深股票专题统计-退市资料下载
## 导出退市公司信息后删除其他列，仅保留代码与退市时间，另存为csv
## 根据代码开头，剔除B股公司，标注上市板块
## 在2019年以后，共有38家退市公司
## 由于退市公司普遍曾被ST，我们实际上不用找回样本
DLcorp = pd.read_csv('Raw Transaction Data/DLcodes.csv', encoding = "utf-8")
DLcorp['DLdate'] = pd.to_datetime(DLcorp['DLdate'])
DLcorp = DLcorp[DLcorp['DLdate'] > '2019-01-01']
DLcorp = DLcorp.reset_index(drop = True)
DLcorp = DLcorp[['DLcodes', 'boardtype']]
DLcorp.columns = ['codes', 'boardtype']

In [12]:
## ST公司信息通过wind沪深股票专题统计-实施ST下载
## 导出ST数据后删除其他列，仅保留代码，另存为csv
## 共有619家公司曾于2019年以前被ST
STcorp = pd.read_csv('Raw Transaction Data/STcodes.csv', encoding = "utf-8")
STcorp['STdate'] = pd.to_datetime(STcorp['STdate'])
STcorp = STcorp[STcorp['STdate'] < '2019-01-01']
STcorp = STcorp.groupby(STcorp['STcodes'])['STdate'].first()
STcorp = STcorp.reset_index()

In [14]:
corps = pd.concat([corp_bf18, DLcorp], ignore_index=True)
codes = corps['codes'].tolist()

In [53]:
corps.to_csv('Raw Transaction Data/AllCorps.csv', index=False, encoding = "utf_8_sig")

## 使用wind下载股票日交易数据

In [17]:
## 启动wind插件
w.start() 
w.isconnected() # 判断WindPy是否已经登录成功

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2020 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [18]:
## 提取2007年的数据
## 将数据转为DF格式，转置，并提取index
wsddata_07 = w.wsd( codes, "pct_chg", "2007-01-01", "2007-12-31", "Currency=CNY;PriceAdj=F")
transdata_07 = pd.DataFrame(wsddata_07.Data,index = wsddata_07.Codes,columns = wsddata_07.Times)
transdata_07 = transdata_07.T #转置
transdata_07 = transdata_07.reset_index()

In [19]:
## 提取2008年的数据
wsddata_08 = w.wsd( codes, "pct_chg", "2008-01-01", "2008-12-31", "Currency=CNY;PriceAdj=F")
transdata_08 = pd.DataFrame(wsddata_08.Data,index = wsddata_08.Codes,columns = wsddata_08.Times)
transdata_08 = transdata_08.T #转置
transdata_08 = transdata_08.reset_index()

In [20]:
## 提取2009年的数据
wsddata_09 = w.wsd( codes, "pct_chg", "2009-01-01", "2009-12-31", "Currency=CNY;PriceAdj=F")
transdata_09 = pd.DataFrame(wsddata_09.Data,index = wsddata_09.Codes,columns = wsddata_09.Times)
transdata_09 = transdata_09.T #转置
transdata_09 = transdata_09.reset_index()

In [21]:
## 提取2010年的数据
wsddata_10 = w.wsd( codes, "pct_chg", "2010-01-01", "2010-12-31", "Currency=CNY;PriceAdj=F")
transdata_10 = pd.DataFrame(wsddata_10.Data,index = wsddata_10.Codes,columns = wsddata_10.Times)
transdata_10 = transdata_10.T #转置
transdata_10 = transdata_10.reset_index()

In [22]:
## 提取2011年的数据
wsddata_11 = w.wsd( codes, "pct_chg", "2011-01-01", "2011-12-31", "Currency=CNY;PriceAdj=F")
transdata_11 = pd.DataFrame(wsddata_11.Data,index = wsddata_11.Codes,columns = wsddata_11.Times)
transdata_11 = transdata_11.T #转置
transdata_11 = transdata_11.reset_index()

In [23]:
## 提取2012年的数据
wsddata_12 = w.wsd( codes, "pct_chg", "2012-01-01", "2012-12-31", "Currency=CNY;PriceAdj=F")
transdata_12 = pd.DataFrame(wsddata_12.Data,index = wsddata_12.Codes,columns = wsddata_12.Times)
transdata_12 = transdata_12.T #转置
transdata_12 = transdata_12.reset_index()

In [24]:
## 提取2013年的数据
wsddata_13 = w.wsd( codes, "pct_chg", "2013-01-01", "2013-12-31", "Currency=CNY;PriceAdj=F")
transdata_13 = pd.DataFrame(wsddata_13.Data,index = wsddata_13.Codes,columns = wsddata_13.Times)
transdata_13 = transdata_13.T #转置
transdata_13 = transdata_13.reset_index()

In [25]:
## 提取2014年的数据
wsddata_14 = w.wsd( codes, "pct_chg", "2014-01-01", "2014-12-31", "Currency=CNY;PriceAdj=F")
transdata_14 = pd.DataFrame(wsddata_14.Data,index = wsddata_14.Codes,columns = wsddata_14.Times)
transdata_14 = transdata_14.T #转置
transdata_14 = transdata_14.reset_index()

In [26]:
## 提取2015年的数据
wsddata_15 = w.wsd( codes, "pct_chg", "2015-01-01", "2015-12-31", "Currency=CNY;PriceAdj=F")
transdata_15 = pd.DataFrame(wsddata_15.Data,index = wsddata_15.Codes,columns = wsddata_15.Times)
transdata_15 = transdata_15.T #转置
transdata_15 = transdata_15.reset_index()

In [27]:
## 提取2016年的数据
wsddata_16 = w.wsd( codes, "pct_chg", "2016-01-01", "2016-12-31", "Currency=CNY;PriceAdj=F")
transdata_16 = pd.DataFrame(wsddata_16.Data,index = wsddata_16.Codes,columns = wsddata_16.Times)
transdata_16 = transdata_16.T #转置
transdata_16 = transdata_16.reset_index()

In [28]:
## 提取2017年的数据
wsddata_17 = w.wsd( codes, "pct_chg", "2017-01-01", "2017-12-31", "Currency=CNY;PriceAdj=F")
transdata_17 = pd.DataFrame(wsddata_17.Data,index = wsddata_17.Codes,columns = wsddata_17.Times)
transdata_17 = transdata_17.T #转置
transdata_17 = transdata_17.reset_index()

In [29]:
## 提取2018年的数据
wsddata_18 = w.wsd( codes, "pct_chg", "2018-01-01", "2018-12-31", "Currency=CNY;PriceAdj=F")
transdata_18 = pd.DataFrame(wsddata_18.Data,index = wsddata_18.Codes,columns = wsddata_18.Times)
transdata_18 = transdata_18.T #转置
transdata_18 = transdata_18.reset_index()

In [30]:
## 合并2007-2018年的交易数据
transdt = pd.concat([transdata_07, transdata_08, transdata_09, transdata_10, transdata_11, 
        transdata_12, transdata_13, transdata_14, transdata_15, transdata_16, transdata_17, transdata_18], ignore_index=True)
transdt.to_csv('Raw Transaction Data/transdt.csv', index=False)

In [31]:
## 读取transdt
transdt = pd.read_csv('Raw Transaction Data/transdt.csv')

## 读取shibor数据

In [19]:
## 提取一月期SHIBOR数据并保存
shibor = w.wsd("SHIBORON.IR", "open", "2007-01-01", "2018-12-31")
shibor = pd.DataFrame(shibor.Data,index = shibor.Codes,columns = shibor.Times)
shibor = shibor.T
shibor = shibor.reset_index()
shibor.columns = ['date', 'shibor']
shibor.to_csv('Raw Transaction Data/shibor.csv', index=False)

In [32]:
## 读取已保存的shibor数据
shibor = pd.read_csv('Raw Transaction Data/shibor.csv')
shibor['shibor'] = shibor['shibor'] * 0.01
shibor['date'] = pd.to_datetime(shibor['date'])

## 匹配个股日收益率与shibor日数据，计算超额收益率

In [33]:
## 将二维表转为一维表，准备分组计算
## 重命名日期列为date，并调整数据格式,提取年、月
transdt_T = pd.melt(transdt, 
            id_vars = 'index', 
            value_vars =list(transdt.columns[1:]), 
            var_name='code', 
            value_name='pct_chg')
transdt_T.columns = ['date', 'code', 'pct_chg']
transdt_T['date'] = pd.to_datetime(transdt_T['date'])
transdt_T['year'] = transdt_T['date'].dt.year
transdt_T['month'] = transdt_T['date'].dt.month
transdt_T['pct_chg'] = transdt_T['pct_chg'] * 0.01
transdt_T = transdt_T[['code', 'date', 'year', 'month', 'pct_chg']]

In [34]:
## 删除当日无交易的数据
transdt_T = transdt_T.dropna(subset = ['pct_chg'])

In [35]:
## 合并债券收益率和shibor，计算超额收益率
exrfore = pd.merge(transdt_T, shibor, on = 'date', how = 'left')
exrfore['exr'] = exrfore['pct_chg'] - exrfore['shibor']

In [36]:
## group by代码和月份，计算各组非NA值的个数、平均收益率，收益率方差、偏度
V_sta = exrfore.groupby([exrfore['code'], exrfore['year'], exrfore['month']])['exr'].agg([('num', 'count'), ('mean', 'mean'), 
        ('math_var', 'var'), ('math_skew', 'skew'),('math_kurt', stats.kurtosis),])
V_sta = V_sta.reset_index()

## 计算调整后的方差、偏度与峰度

In [37]:
## 提取各月第一行，仅保留code与date，插入flag列，准备剔除
exrlinef = exrfore.groupby([exrfore['code'], exrfore['year'], exrfore['month']])['date'].first()
exrlinef = exrlinef.reset_index()
exrlinef = exrlinef[['code', 'date']]
exrlinef.insert(2, 'flag', np.ones(len(exrlinef)))

In [38]:
## 剔除第一行，并重命名列, wof指without first line
exr_wof = pd.merge(exrfore, exrlinef, on = ['code', 'date'], how = 'left')
exr_wof = exr_wof[exr_wof['flag'] != 1]
exr_wof = exr_wof[['code', 'date', 'exr']]
exr_wof = exr_wof.rename(columns={'exr': 'exr_f'})
exr_wof.insert(1, 'order', np.arange(len(exr_wof)))

In [39]:
## 提取各月最后一行，仅保留code与date，插入flag列，准备剔除
exrlinel = exrfore.groupby([exrfore['code'], exrfore['year'], exrfore['month']])['date'].last()
exrlinel = exrlinel.reset_index()
exrlinel = exrlinel[['code', 'date']]
exrlinel.insert(2, 'flag', np.ones(len(exrlinef)))

In [40]:
## 剔除最后一行，并重命名列, wol指without last line
exr_wol = pd.merge(exrfore, exrlinel, on = ['code', 'date'], how = 'left')
exr_wol = exr_wol[exr_wol['flag'] != 1]
exr_wol = exr_wol[['code', 'year', 'month', 'exr']]
exr_wol = exr_wol.rename(columns={'exr': 'exr_l'})
exr_wol.insert(1, 'order', np.arange(len(exr_wol)))

In [41]:
## 合并去除第一行/最后一行的数据，准备计算调整的方差
## 注意，保留的日期系exr_f的日期
calvar0 = pd.merge(exr_wol, exr_wof, on = ['code', 'order'], how = 'left')
calvar0 = calvar0[['code', 'date', 'year', 'month', 'exr_f', 'exr_l']]

In [42]:
## 为calvar匹配mean
calvar = pd.merge(calvar0, V_sta[['code', 'year', 'month', 'mean']], on = ['code', 'year', 'month'], how = 'left')
calvar['diff1'] = calvar['exr_f'] - calvar['mean']
calvar['diff2'] = calvar['exr_l'] - calvar['mean']
calvar['product1'] = calvar['diff1'] * calvar['diff2']
calvar = calvar.groupby([calvar['code'], calvar['year'], calvar['month']])['product1'].agg([('prdct1', 'sum'),])
calvar = calvar.reset_index()

In [43]:
calvar.to_csv('Raw Transaction Data/calvar.csv', index=False)

In [44]:
V_sta.to_csv('Raw Transaction Data/vsta.csv', index=False)

In [45]:
## 计算调整后的方差、偏度、峰度
V_sta = pd.merge(V_sta, calvar[['code', 'year', 'month', 'prdct1']], on = ['code', 'year', 'month'], how = 'left' )
V_sta['var'] = V_sta['math_var'] * V_sta['num'] + 2 * V_sta['prdct1']
V_sta['adj_var'] = V_sta['var']/V_sta['num']

In [47]:
## 在任意月，保留交易日数量大于10的样本
prdct_dt = V_sta[V_sta['num'] >= 10]
prdct_dt = prdct_dt[['code', 'year', 'month', 'num', 'var', 'adj_var', 'math_skew', 'math_kurt']]

In [48]:
## 保存var, skew, kurt数据
prdct_dt.to_csv('Raw Transaction Data/prdct_dt.csv', index=False)

In [50]:
prdct_dt

,code,year,month,num,var,adj_var,math_skew,math_kurt
0,000001.SZ,2007,1,20,0.046588,0.002329,-0.897868,-0.690205
1,000001.SZ,2007,2,15,0.027840,0.001856,0.168187,-1.414870
2,000001.SZ,2007,3,22,0.009971,0.000453,0.366919,-0.376840
3,000001.SZ,2007,4,21,0.025306,0.001205,-0.734114,-0.084518
4,000001.SZ,2007,5,18,0.003345,0.000186,0.307989,-0.024779
...,...,...,...,...,...,...,...,...
338306,603999.SH,2018,8,23,0.011140,0.000484,0.517311,-0.263385
338307,603999.SH,2018,9,19,0.002344,0.000123,-0.664188,-0.413938
338308,603999.SH,2018,10,18,0.073409,0.004078,0.159128,0.074828
338309,603999.SH,2018,11,22,0.015331,0.000697,-0.131413,-0.705213
